# 🔄 Evaluator-Optimizer Multi-Agent Graph

This notebook demonstrates how to build an **Evaluator-Optimizer Workflow** using the **OpenAI Agents SDK** with **Groq**.

### 🎯 Workflow Topology
1. **🟢 Writer Node (Optimizer)**: Generates initial drafts or refines content based on reviewer feedback. Mutates graph state via `save_draft` tool.
2. **🟢 Reviewer Node (Evaluator / Decision Gate)**: Evaluates quality, clarity, and constraints. If criteria are met or max iterations reached, hands off to the **Publisher Node**. Otherwise, records feedback via `record_feedback` tool and loops back to **Writer Node**.
3. **🟢 Publisher Node (Exit Gate)**: Produces the final structured output conforming to a Pydantic `EvaluationReport` model.

## 1. Setup & Environment Configuration

In [ ]:
import os
import sys
import asyncio
from dataclasses import dataclass, field
from dotenv import load_dotenv
import openai
from pydantic import BaseModel, Field

from agents import (
    Agent,
    Runner,
    OpenAIChatCompletionsModel,
    ModelSettings,
    RunContextWrapper,
    function_tool,
    handoff,
    set_tracing_disabled,
)

# Load environment variables from .env
load_dotenv(override=True)

# Disable OpenAI tracing to avoid 429 quota errors on non-OpenAI endpoints
os.environ.pop("OPENAI_API_KEY", None)
set_tracing_disabled(True)

# Setup Groq Async client
client = openai.AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY"),
)

model = OpenAIChatCompletionsModel(
    model="qwen/qwen3.6-27b",
    openai_client=client,
)

def sanitize_schema_for_groq(schema_dict: dict):
    """Ensure JSON schemas adhere strictly to Groq API validation rules."""
    if not schema_dict.get("properties"):
        schema_dict.pop("required", None)
    else:
        schema_dict["required"] = list(schema_dict["properties"].keys())

print("✅ Environment initialized and Groq model configured.")

## 2. Graph State & Structured Output Schemas

In [ ]:
# --- Shared Graph Context ---
@dataclass
class WorkflowState:
    """Shared state tracked across all nodes in the evaluation graph."""
    topic: str
    current_draft: str = ""
    feedback_notes: list[str] = field(default_factory=list)
    iteration: int = 0
    max_iterations: int = 3
    is_approved: bool = False

# --- Final Structured Exit Contract ---
class EvaluationReport(BaseModel):
    """The final approved payload produced at the graph exit node."""
    title: str = Field(description="Catchy title of the final content")
    final_content: str = Field(description="The finalized, polished content")
    quality_score: int = Field(description="Score from 1 to 100")
    total_iterations: int = Field(description="Number of draft-review cycles taken")
    review_summary: str = Field(description="Brief summary of why this draft was approved")

print("✅ Schemas defined: WorkflowState (Context) & EvaluationReport (Pydantic Output).")

## 3. State Mutation Function Tools

In [ ]:
@function_tool
def save_draft(ctx: RunContextWrapper[WorkflowState], draft_text: str) -> str:
    """Save the newly generated or revised draft into the shared graph state."""
    state = ctx.context
    state.iteration += 1
    state.current_draft = draft_text
    return f"Draft #{state.iteration} saved to graph state: {draft_text}"

@function_tool
def record_feedback(ctx: RunContextWrapper[WorkflowState], feedback: str) -> str:
    """Record reviewer feedback into graph state to guide the next writing iteration."""
    state = ctx.context
    state.feedback_notes.append(feedback)
    return f"Feedback recorded: {feedback}"

# Sanitize tool schemas for Groq compatibility
sanitize_schema_for_groq(save_draft.params_json_schema)
sanitize_schema_for_groq(record_feedback.params_json_schema)

print("✅ State management tools registered and sanitized for Groq.")

## 4. Graph Nodes & Directed Edges (Handoffs)

In [ ]:
# Node A: Writer Node (Optimizer)
writer_node = Agent(
    name="Writer Node",
    instructions=(
        "You are the Writer Node in an Evaluator-Optimizer graph.\n"
        "Task: Generate or refine a concise 2-sentence draft based on the topic and any reviewer feedback.\n"
        "Step 1: Call `save_draft` with the draft text.\n"
        "Step 2: Immediately hand off to Reviewer Node."
    ),
    tools=[save_draft],
    model=model,
    model_settings=ModelSettings(max_tokens=350),
)

# Node C: Publisher Node (Exit / Formatter Node with Structured Output)
publisher_node = Agent(
    name="Publisher Node",
    instructions=(
        "You are the Publisher Node. Review the latest draft from the workflow and produce "
        "the final EvaluationReport structured output."
    ),
    output_type=EvaluationReport,
    model=model,
    model_settings=ModelSettings(max_tokens=400),
)

# Node B: Reviewer Node (Evaluator / Decision Gate)
reviewer_node = Agent(
    name="Reviewer Node",
    instructions=(
        "You are the Reviewer / Evaluator Node.\n"
        "Evaluate the draft in context (score 1-100 for clarity, punchiness, and constraints).\n"
- If score >= 85 or if a revision was already made: Hand off immediately to Publisher Node.\n"
- If score < 85: First call `record_feedback` with specific improvement advice, then hand off to Writer Node."
    ),
    tools=[record_feedback],
    model=model,
    model_settings=ModelSettings(max_tokens=350),
)

# Define Graph Directed Edges
to_reviewer = handoff(reviewer_node)
sanitize_schema_for_groq(to_reviewer.input_json_schema)

to_writer = handoff(writer_node)
sanitize_schema_for_groq(to_writer.input_json_schema)

to_publisher = handoff(publisher_node)
sanitize_schema_for_groq(to_publisher.input_json_schema)

# Wire Graph Topology
writer_node.handoffs = [to_reviewer]
reviewer_node.handoffs = [to_writer, to_publisher]

print("✅ Graph nodes and edges wired successfully:")
print("   Writer Node -> Reviewer Node <-> Writer Node (Loop) -> Publisher Node (Exit)")

## 5. Execution Pipeline with Live Streaming

In [ ]:
async def run_evaluator_optimizer_graph(topic_prompt: str):
    print("=" * 70)
    print("🚀 STARTING EVALUATOR-OPTIMIZER AGENT GRAPH")
    print(f"📌 Goal: {topic_prompt}")
    print("=" * 70)

    state = WorkflowState(topic=topic_prompt)

    result = Runner.run_streamed(
        starting_agent=writer_node,
        input=f"Goal: {topic_prompt}",
        context=state,
        max_turns=25,
    )

    current_node = "Writer Node"
    print(f"\n🟢 [Node: {current_node}] Activated")

    async for event in result.stream_events():
        # Edge transitions
        if event.type == "agent_updated_stream_event":
            current_node = event.new_agent.name
            print(f"\n\n🔀 [Graph Edge] Transitioning -> 🟢 {current_node}")

        # Tool execution
        elif event.type == "run_item_stream_event":
            if event.name == "tool_called":
                tool_name = getattr(event.item, "tool_name", "tool")
                print(f"\n  ⚙️  [{current_node}] Calling: {tool_name}...")
            elif event.name == "tool_output":
                output = getattr(event.item, "output", "")
                print(f"  ✅ [{current_node}] Result: {output}")

        # Token streaming
        elif event.type == "raw_response_event":
            if hasattr(event.data, "delta") and event.data.delta:
                print(event.data.delta, end="", flush=True)

    print("\n\n" + "=" * 70)
    print("🏁 GRAPH WORKFLOW COMPLETED")
    print("=" * 70)

    report: EvaluationReport = result.final_output
    if report and isinstance(report, EvaluationReport):
        print(f"\n📊 Final Evaluator Report (Structured Pydantic Data):")
        print(f"  • Title             : {report.title}")
        print(f"  • Quality Score     : {report.quality_score}/100")
        print(f"  • Total Iterations  : {report.total_iterations}")
        print(f"  • Reviewer Summary  : {report.review_summary}")
        print(f"\n📝 Final Approved Copy:\n{report.final_content}")
    else:
        print(f"Final output: {report}")

    print("\n" + "-" * 70)
    print("🏦 Final Python Graph State:")
    print(f"  • Total Draft Cycles: {state.iteration}")
    print(f"  • Feedback History  : {len(state.feedback_notes)} items")
    for idx, fb in enumerate(state.feedback_notes, 1):
        print(f"    {idx}. {fb}")
    print("-" * 70)
    return report, state

## 6. Interactive Execution Cell

In [ ]:
# In Jupyter / IPython notebooks, top-level await is directly supported:
task_prompt = "Write a compelling 2-sentence launch announcement for an open-source AI agent framework."
report, final_state = await run_evaluator_optimizer_graph(task_prompt)